<a href="https://colab.research.google.com/github/jaspreet-aidev/RiceDoctor-EdgeAI/blob/main/training_model_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. AGGRESSIVE DATA AUGMENTATION (Forces general feature learning)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # Automatically reserve 20% for validation
)

# Training Data Generator
# IMPORTANT: Replace 'path_to_your_dataset_folder' with your actual dataset path
train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/kisan_mitra_data/Rice_Leaf_AUG',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# Validation Data Generator (No augmentation except rescaling)
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

val_generator = val_datagen.flow_from_directory(
    '/content/drive/MyDrive/kisan_mitra_data/Rice_Leaf_AUG',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# 2. HARDENED ARCHITECTURE (MobileNetV2 Base + Dropout)
baseModel = MobileNetV2(weights="imagenet", include_top=False,
                        input_tensor=Input(shape=(224, 224, 3)))

# Freeze base model layers so we train the classification head first
for layer in baseModel.layers:
    layer.trainable = False

headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(256, activation="relu")(headModel)
headModel = Dropout(0.5)(headModel)  # Hardening Dropout to prevent memorization
headModel = Dense(train_generator.num_classes, activation="softmax")(headModel)

model = Model(inputs=baseModel.input, outputs=headModel)

# Compile the model
opt = Adam(learning_rate=1e-4)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

# 3. EARLY STOPPING CALLBACK (Stops overfitting instantly)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_rice_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Execute Training Loop
print("[INFO] Starting Hardened Training Loop...")
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=val_generator,
    validation_steps=val_generator.samples // 32,
    epochs=30,
    callbacks=callbacks
)

Found 3066 images belonging to 6 classes.
Found 763 images belonging to 6 classes.


/tmp/ipykernel_978/2162374348.py:47: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  baseModel = MobileNetV2(weights="imagenet", include_top=False,


[INFO] Starting Hardened Training Loop...
Epoch 1/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 606ms/step - accuracy: 0.2497 - loss: 2.0046
Epoch 1: val_accuracy improved from None to 0.43342, saving model to best_rice_model.h5



Epoch 1: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 94s 883ms/step - accuracy: 0.3174 - loss: 1.7723 - val_accuracy: 0.4334 - val_loss: 1.4449
Epoch 2/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5000 - loss: 1.4619

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


...


Epoch 2: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 19s 199ms/step - accuracy: 0.5000 - loss: 1.4619 - val_accuracy: 0.4416 - val_loss: 1.4390
Epoch 3/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 572ms/step - accuracy: 0.4705 - loss: 1.3808
Epoch 3: val_accuracy did not improve from 0.44158
95/95 ━━━━━━━━━━━━━━━━━━━━ 73s 765ms/step - accuracy: 0.4974 - loss: 1.3352 - val_accuracy: 0.4307 - val_loss: 1.4025
Epoch 4/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.5312 - loss: 1.1140
Epoch 4: val_accuracy did not improve from 0.44158
95/95 ━━━━━━━━━━━━━━━━━━━━ 21s 222ms/step - accuracy: 0.5312 - loss: 1.1140 - val_accuracy: 0.4389 - val_loss: 1.3897
Epoch 5/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 556ms/step - accuracy: 0.5356 - loss: 1.2091
Epoch 5: val_accuracy improved from 0.44158 to 0.44293, saving model to best_rice_model.h5



Epoch 5: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 74s 769ms/step - accuracy: 0.5465 - loss: 1.1914 - val_accuracy: 0.4429 - val_loss: 1.3373
Epoch 6/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.6562 - loss: 1.0398
Epoch 6: val_accuracy improved from 0.44293 to 0.44837, saving model to best_rice_model.h5



Epoch 6: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 19s 197ms/step - accuracy: 0.6562 - loss: 1.0398 - val_accuracy: 0.4484 - val_loss: 1.3460
Epoch 7/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 579ms/step - accuracy: 0.5778 - loss: 1.1181
Epoch 7: val_accuracy improved from 0.44837 to 0.48777, saving model to best_rice_model.h5



Epoch 7: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 82s 868ms/step - accuracy: 0.5811 - loss: 1.1091 - val_accuracy: 0.4878 - val_loss: 1.2736
Epoch 8/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.6875 - loss: 1.0443
Epoch 8: val_accuracy improved from 0.48777 to 0.49457, saving model to best_rice_model.h5



Epoch 8: finished saving model to best_rice_model.h5
95/95 ━━━━━━━━━━━━━━━━━━━━ 19s 201ms/step - accuracy: 0.6875 - loss: 1.0443 - val_accuracy: 0.4946 - val_loss: 1.2611
Epoch 9/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 574ms/step - accuracy: 0.6224 - loss: 1.0365
Epoch 9: val_accuracy did not improve from 0.49457
95/95 ━━━━━━━━━━━━━━━━━━━━ 73s 768ms/step - accuracy: 0.6256 - loss: 1.0106 - val_accuracy: 0.4742 - val_loss: 1.3063
Epoch 10/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.5938 - loss: 1.3343
Epoch 10: val_accuracy did not improve from 0.49457
95/95 ━━━━━━━━━━━━━━━━━━━━ 19s 203ms/step - accuracy: 0.5938 - loss: 1.3343 - val_accuracy: 0.4688 - val_loss: 1.3266
Epoch 11/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 549ms/step - accuracy: 0.6251 - loss: 1.0069
Epoch 11: val_accuracy did not improve from 0.49457
95/95 ━━━━━━━━━━━━━━━━━━━━ 72s 756ms/step - accuracy: 0.6295 - loss: 0.9905 - val_accuracy: 0.4552 - val_loss: 1.3665
Epoch 12/30
 1/95 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accura